In [2]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
from sklearn.model_selection import train_test_split
from concurrent.futures import ProcessPoolExecutor, as_completed

def extract_comprehensive_features(file_path: str, top_db: int = 30, n_mfcc: int = 13) -> dict:

    y, sr = librosa.load(file_path, sr=None, mono=True)
    y_trimmed, _ = librosa.effects.trim(y, top_db=top_db)

    feats = {}
    # Basic features
    feats["duration"] = float(librosa.get_duration(y=y, sr=sr))
    feats["rms_mean"] = float(np.mean(librosa.feature.rms(y=y_trimmed)))
    feats["zcr_mean"] = float(np.mean(librosa.feature.zero_crossing_rate(y_trimmed)))

    # Spectral
    sc = librosa.feature.spectral_centroid(y=y_trimmed, sr=sr)
    sr0 = librosa.feature.spectral_rolloff(y=y_trimmed, sr=sr)
    sbw = librosa.feature.spectral_bandwidth(y=y_trimmed, sr=sr)
    scon = librosa.feature.spectral_contrast(y=y_trimmed, sr=sr)

    feats["spectral_centroid_mean"] = float(np.mean(sc))
    feats["spectral_centroid_std"]  = float(np.std(sc))
    feats["spectral_rolloff_mean"]  = float(np.mean(sr0))
    feats["spectral_bandwidth_mean"] = float(np.mean(sbw))
    feats["spectral_contrast_mean"] = float(np.mean(scon))

    # MFCC (mean/std per coeff)
    mfcc = librosa.feature.mfcc(y=y_trimmed, sr=sr, n_mfcc=n_mfcc)
    for i in range(n_mfcc):
        feats[f"mfcc_{i+1}_mean"] = float(np.mean(mfcc[i]))
        feats[f"mfcc_{i+1}_std"]  = float(np.std(mfcc[i]))

    # Chroma
    chroma = librosa.feature.chroma_stft(y=y_trimmed, sr=sr)
    feats["chroma_mean"] = float(np.mean(chroma))
    feats["chroma_std"]  = float(np.std(chroma))

    # Tempo
    try:
        tempo, _ = librosa.beat.beat_track(y=y_trimmed, sr=sr)
        feats["tempo"] = float(tempo)
    except Exception:
        feats["tempo"] = 0.0

    # Pitch
    try:
        f0, voiced_flag, _ = librosa.pyin(
            y_trimmed,
            fmin=librosa.note_to_hz("C2"),
            fmax=librosa.note_to_hz("C7"),
        )
        # voiced frames only
        f0_voiced = f0[voiced_flag] if f0 is not None else np.array([])
        if len(f0_voiced) > 0:
            feats["pitch_mean"] = float(np.mean(f0_voiced))
            feats["pitch_std"]  = float(np.std(f0_voiced))
            feats["pitch_range"] = float(np.max(f0_voiced) - np.min(f0_voiced))
        else:
            feats["pitch_mean"] = feats["pitch_std"] = feats["pitch_range"] = 0.0

        feats["voicing_rate"] = float(np.sum(voiced_flag) / len(voiced_flag)) if len(voiced_flag) else 0.0
    except Exception:
        feats["pitch_mean"] = feats["pitch_std"] = feats["pitch_range"] = feats["voicing_rate"] = 0.0

    return feats


# Canonical emotion mapping

CANON = {"angry", "happy", "sad", "neutral", "fear", "disgust"}

def to_canonical(emotion: str) -> str | None:
    """
    Map dataset-specific labels into canonical set.
    Return None to drop.
    """
    e = emotion.strip().lower()

    mapping = {
        "anger": "angry",
        "ang": "angry",
        "rab": "angry",

        "happiness": "happy",
        "hap": "happy",
        "joy": "happy",
        "excited": "happy",
        "gio": "happy",

        "sadness": "sad",
        "sad": "sad",
        "tri": "sad",

        "neutral": "neutral",
        "neu": "neutral",
        "calm": "neutral",

        "fear": "fear",
        "fea": "fear",
        "fearful": "fear",
        "pau": "fear",

        "disgust": "disgust",
        "dis": "disgust",
        "disg": "disgust",
        "disgusto": "disgust",
    }

    if e in ("surprise", "surprised", "ps", "sur"):
        return None

    out = mapping.get(e, e)
    return out if out in CANON else None

def parse_cremad(p: Path) -> dict | None:
    parts = p.stem.split("_")
    if len(parts) < 4:
        return None
    speaker_id, sentence_id, emo, intensity = parts[0], parts[1], parts[2], parts[3]
    emo2 = to_canonical(emo)
    if emo2 is None:
        return None
    return {
        "filepath": str(p),
        "speaker_id": speaker_id,
        "utterance_id": sentence_id,
        "emotion": emo2,
        "emotion_raw": emo,
        "intensity": intensity,
        "dataset": "CREMA-D",
        "language": "en",
    }


def parse_tess(p: Path) -> dict | None:
    stem = p.stem
    parts = stem.split("_")
    if len(parts) < 2:
        return None

    speaker_id = parts[0]
    emo_raw = parts[-1]
    emo2 = to_canonical(emo_raw)
    if emo2 is None:
        return None

    utterance_id = "_".join(parts[1:-1]) if len(parts) > 2 else "utt"

    return {
        "filepath": str(p),
        "speaker_id": speaker_id,
        "utterance_id": utterance_id,
        "emotion": emo2,
        "emotion_raw": emo_raw,
        "intensity": "",
        "dataset": "TESS",
        "language": "en",
    }


def parse_ravdess(p: Path) -> dict | None:
    m = re.match(r"(\d{2})-(\d{2})-(\d{2})-(\d{2})-(\d{2})-(\d{2})-(\d{2})$", p.stem)
    if not m:
        return None

    emotion_code = int(m.group(3))
    intensity_code = int(m.group(4))
    actor = m.group(7)

    emotion_map = {
        1: "neutral",
        2: "neutral",
        3: "happy",
        4: "sad",
        5: "angry",
        6: "fear",
        7: "disgust",
        8: None,
    }

    emo2 = emotion_map.get(emotion_code, None)
    if emo2 is None:
        return None

    return {
        "filepath": str(p),
        "speaker_id": actor,
        "utterance_id": p.stem,
        "emotion": emo2,
        "emotion_raw": str(emotion_code),
        "intensity": str(intensity_code),
        "dataset": "RAVDESS",
        "language": "en",
    }


def parse_emovo(p: Path) -> dict | None:
    # Example: dis-f1-b1.wav
    parts = p.stem.lower().split("-")
    if len(parts) < 3:
        return None

    emo_raw = parts[0] 
    speaker_id = parts[1]
    utt_id = "-".join(parts[2:])

    emo2 = to_canonical(emo_raw)
    if emo2 is None:
        return None

    return {
        "filepath": str(p),
        "speaker_id": speaker_id,
        "utterance_id": utt_id,
        "emotion": emo2,
        "emotion_raw": emo_raw,
        "intensity": "",
        "dataset": "EMOVO",
        "language": "it",
    }


PARSERS = {
    "cremad": parse_cremad,
    "tess": parse_tess,
    "ravdess": parse_ravdess,
    "emovo": parse_emovo,
}

def process_one(path_str: str, dataset_key: str, top_db: int, n_mfcc: int) -> dict | None:
    p = Path(path_str)
    rec = PARSERS[dataset_key](p)
    if rec is None:
        return None
    feats = extract_comprehensive_features(path_str, top_db=top_db, n_mfcc=n_mfcc)
    rec.update(feats)
    return rec

def build_dataset_csv(
    dataset_key: str,
    raw_dir: str,
    out_csv: str,
    n_mfcc: int = 13,
    top_db: int = 30,
    n_jobs: int = 1,
    seed: int = 42,
    make_speaker_split: bool = True,
):
    dataset_key = dataset_key.lower()
    if dataset_key not in PARSERS:
        raise ValueError(f"Unknown dataset_key={dataset_key}. Choose from {list(PARSERS)}")

    raw_dir = Path(raw_dir)
    wavs = list(raw_dir.rglob("*.wav"))
    if not wavs:
        raise FileNotFoundError(f"No .wav files found under: {raw_dir}")

    rows = []
    paths = [str(p) for p in wavs]

    if n_jobs == 1:
        for ps in tqdm(paths, desc=f"Extracting {dataset_key}"):
            rec = process_one(ps, dataset_key, top_db, n_mfcc)
            if rec:
                rows.append(rec)
    else:
        with ProcessPoolExecutor(max_workers=n_jobs) as ex:
            futs = [ex.submit(process_one, ps, dataset_key, top_db, n_mfcc) for ps in paths]
            for fut in tqdm(as_completed(futs), total=len(futs), desc=f"Extracting {dataset_key} (mp)"):
                rec = fut.result()
                if rec:
                    rows.append(rec)

    df = pd.DataFrame(rows)

    # speaker-safe split (train/val/test)
    if make_speaker_split:
        speakers = df["speaker_id"].astype(str).unique()
        train_spk, temp_spk = train_test_split(speakers, test_size=0.30, random_state=seed)
        val_spk, test_spk = train_test_split(temp_spk, test_size=0.50, random_state=seed)

        def split_from_speaker(s: str) -> str:
            if s in train_spk:
                return "train"
            if s in val_spk:
                return "val"
            return "test"

        df["split"] = df["speaker_id"].astype(str).apply(split_from_speaker)
    else:
        df["split"] = ""

    out_csv = Path(out_csv)
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_csv, index=False)

    return df


# Build all csv files

def build_all(
    cremad_dir: str,
    tess_dir: str,
    ravdess_dir: str,
    emovo_dir: str,
    out_dir: str = "artifacts",
    n_mfcc: int = 13,
    top_db: int = 30,
    n_jobs: int = 1,
    seed: int = 42,
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    df_cre = build_dataset_csv("cremad", cremad_dir, str(out_dir/"cremad_features.csv"),
                              n_mfcc=n_mfcc, top_db=top_db, n_jobs=n_jobs, seed=seed, make_speaker_split=True)

    df_tes = build_dataset_csv("tess", tess_dir, str(out_dir/"tess_features.csv"),
                               n_mfcc=n_mfcc, top_db=top_db, n_jobs=n_jobs, seed=seed, make_speaker_split=True)

    df_rav = build_dataset_csv("ravdess", ravdess_dir, str(out_dir/"ravdess_features.csv"),
                               n_mfcc=n_mfcc, top_db=top_db, n_jobs=n_jobs, seed=seed, make_speaker_split=True)

    # EMOVO as holdout
    df_emo = build_dataset_csv("emovo", emovo_dir, str(out_dir/"emovo_features.csv"),
                               n_mfcc=n_mfcc, top_db=top_db, n_jobs=n_jobs, seed=seed, make_speaker_split=False)
    df_emo["split"] = "holdout_test"
    df_emo.to_csv(out_dir/"emovo_features.csv", index=False)

    # Combine train datasets (exclude EMOVO)
    combined = pd.concat([df_cre, df_tes, df_rav], ignore_index=True)
    combined.to_csv(out_dir/"combined_train_features.csv", index=False)

    return df_cre, df_tes, df_rav, df_emo, combined



In [3]:
df_emovo = build_dataset_csv(
    dataset_key="emovo",
    raw_dir="EMOVO",
    out_csv="artifacts/emovo_features.csv",
    n_mfcc=13,
    top_db=30,
    n_jobs=1,
    seed=42,
    make_speaker_split=True
)

df_emovo["split"] = "holdout_test"
df_emovo.to_csv("artifacts/emovo_features.csv", index=False)

print(df_emovo.shape)
print(df_emovo["speaker_id"].value_counts())
print(df_emovo["emotion"].value_counts())

Extracting emovo:   0%|                                                                        | 0/588 [00:00<?, ?it/s]C:\Users\Donva\AppData\Local\Temp\ipykernel_15392\3060138865.py:47: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  feats["tempo"] = float(tempo)
Extracting emovo: 100%|██████████████████████████████████████████████████████████████| 588/588 [20:26<00:00,  2.09s/it]

(504, 50)
speaker_id
f1    84
f2    84
f3    84
m1    84
m2    84
m3    84
Name: count, dtype: int64
emotion
disgust    84
happy      84
neutral    84
fear       84
angry      84
sad        84
Name: count, dtype: int64
